# 03 — EDA & Insight Discovery

Khám phá pattern chính: screen time, lifestyle, age, gender, location và các quan hệ hành vi.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

# Define paths
PROJECT_ROOT = Path("..")
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "mobile_usage_behavioral_analysis.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AGG_DIR = PROJECT_ROOT / "data" / "aggregated"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

# Create directories if they don't exist
for p in [PROCESSED_DIR, AGG_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
def title(t):
    print("\n" + "="*90); print(t); print("="*90)
def clean_cols(df):
    df = df.copy(); df.columns = [c.strip() for c in df.columns]; return df

# Load and clean dataset
title("Load processed dataset")
df = pd.read_csv(PROCESSED_DIR / "smartphone_features.csv")
print(df.shape)
display(df.head())


In [ ]:
# Executive metrics and opening hook
title("Executive metrics and opening hook")
total = len(df)
heavy_extreme = df[df["Screen_Time_Segment"].isin(["Heavy (8–12h)", "Extreme (>12h)"])].shape[0]
heavy_rate = heavy_extreme / total * 100
overview = pd.DataFrame({"Metric": ["Total users", "Avg screen time", "Avg total app usage", "Avg apps used", "Heavy/Extreme users"], "Value": [f"{total:,}", f"{df['Daily_Screen_Time_Hours'].mean():.2f}h", f"{df['Total_App_Usage_Hours'].mean():.2f}h", f"{df['Number_of_Apps_Used'].mean():.2f}", f"{heavy_rate:.1f}%"]})
display(overview)

# Screen time segment distribution
segment_order = ["Light (<4h)", "Moderate (4–8h)", "Heavy (8–12h)", "Extreme (>12h)"]
seg = df["Screen_Time_Segment"].value_counts().reindex(segment_order).reset_index()
seg.columns = ["Screen_Time_Segment", "Users"]
fig = px.bar(seg, x="Screen_Time_Segment", y="Users", text="Users", title="Screen time segment distribution")
fig.update_traces(textposition="outside")
fig.show()


In [ ]:
# Daily screen time distribution
title("Distribution analysis")
fig = px.histogram(df, x="Daily_Screen_Time_Hours", nbins=35, marginal="box", color="Screen_Time_Segment", title="Daily screen time distribution")
fig.add_vline(x=8, line_dash="dash", annotation_text="Heavy threshold")
fig.add_vline(x=12, line_dash="dash", annotation_text="Extreme threshold")
fig.show()

# Total app usage and number of apps used distribution
for c in ["Total_App_Usage_Hours", "Number_of_Apps_Used", "Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours"]:
    fig = px.histogram(df, x=c, nbins=30, marginal="box", title=f"Distribution — {c}")
    fig.show()


In [ ]:
# Lifestyle segmentation
title("Lifestyle segmentation")
lifestyle = df["Dominant_Lifestyle"].value_counts().reset_index()
lifestyle.columns = ["Dominant_Lifestyle", "Users"]
lifestyle["Percentage"] = lifestyle["Users"] / len(df) * 100
display(lifestyle)
fig = px.bar(lifestyle.sort_values("Users"), x="Users", y="Dominant_Lifestyle", orientation="h", text=lifestyle.sort_values("Users")["Percentage"].round(1).astype(str)+"%", title="Dominant lifestyle distribution")
fig.update_traces(textposition="outside")
fig.show()

# Average category usage by lifestyle
usage_by_lifestyle = df.groupby("Dominant_Lifestyle", as_index=False)[["Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Daily_Screen_Time_Hours"]].mean().round(2)
display(usage_by_lifestyle)
fig = px.bar(usage_by_lifestyle, x="Dominant_Lifestyle", y=["Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours"], barmode="group", title="Average category usage by lifestyle")
fig.show()


In [ ]:
# Demographics and location
title("Demographics and location")
age_order = ["18–24", "25–34", "35–44", "45–54", "55–59"]
age = df.groupby("Age_Group", as_index=False)[["Daily_Screen_Time_Hours", "Total_App_Usage_Hours", "Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours"]].mean().round(2)
age["Age_Group"] = pd.Categorical(age["Age_Group"], categories=age_order, ordered=True)
age = age.sort_values("Age_Group")
loc = df.groupby("Location", as_index=False)["Daily_Screen_Time_Hours"].mean().sort_values("Daily_Screen_Time_Hours", ascending=False)
gender = df.groupby("Gender", as_index=False)["Daily_Screen_Time_Hours"].mean()
display(age); display(loc.round(2)); display(gender.round(2))

# Visualize demographics and location insights
px.line(age, x="Age_Group", y="Daily_Screen_Time_Hours", markers=True, title="Average screen time by age group").show()
fig = px.bar(loc.sort_values("Daily_Screen_Time_Hours"), x="Daily_Screen_Time_Hours", y="Location", orientation="h", text="Daily_Screen_Time_Hours", title="Average screen time by location")
fig.update_traces(texttemplate="%{text:.2f}h", textposition="outside")
fig.show()
px.bar(gender, x="Gender", y="Daily_Screen_Time_Hours", text="Daily_Screen_Time_Hours", title="Average screen time by gender").show()


In [ ]:
# Heatmap and relationship analysis
title("Heatmap and relationship analysis")
heatmap_cols = ["Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Total_App_Usage_Hours", "Daily_Screen_Time_Hours"]
fig = px.imshow(age.set_index("Age_Group")[heatmap_cols], text_auto=True, aspect="auto", title="Average usage hours by age group", labels=dict(color="Hours"))
fig.show()

# Scatter plots of key relationships
pairs = [("Number_of_Apps_Used", "Daily_Screen_Time_Hours"), ("Total_App_Usage_Hours", "Daily_Screen_Time_Hours"), ("Social_Media_Usage_Hours", "Daily_Screen_Time_Hours"), ("Productivity_App_Usage_Hours", "Daily_Screen_Time_Hours"), ("Gaming_App_Usage_Hours", "Daily_Screen_Time_Hours")]
for x, y in pairs:
    fig = px.scatter(df, x=x, y=y, color="Dominant_Lifestyle", opacity=0.55, trendline="ols", hover_data=["Age", "Gender", "Location"], title=f"{x} vs {y}")
    fig.show()

# Correlation matrix
corr_cols = ["Age", "Total_App_Usage_Hours", "Daily_Screen_Time_Hours", "Number_of_Apps_Used", "Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Entertainment_Hours", "App_Fragmentation_Score"]
fig = px.imshow(df[corr_cols].corr().round(2), text_auto=True, aspect="auto", title="Correlation matrix")
fig.show()


In [ ]:
# Key insights export
title("Key insights export")
insights = pd.DataFrame([
    ["Heavy screen time is common", f"{heavy_rate:.1f}% of users are Heavy or Extreme users.", "Hero KPI + highlighted distribution"],
    ["Digital lifestyles are balanced", "Social, productivity and gaming dominant groups are close in size.", "Lifestyle segmentation page"],
    ["Location differences are visible", f"{loc.iloc[0]['Location']} has the highest average screen time at {loc.iloc[0]['Daily_Screen_Time_Hours']:.2f}h.", "Location comparison"],
    ["App count alone is not enough", "Apps used has some relationship with screen time but does not fully explain it.", "Scatter + density contour"],
    ["Category behavior adds context", "Similar screen time may correspond to different social/productivity/gaming profiles.", "Cluster and PCA analysis"],
], columns=["Insight", "Evidence", "Dashboard Use"])
display(insights)
insights.to_csv(PROCESSED_DIR / "eda_key_insights.csv", index=False)
